# SenseVoice 数据量消融实验

验证不同数据量（25%/50%/75%）和两种数据增强策略对微调效果的影响。

In [ ]:
import torch
import torchaudio
import os
import json
import random
import numpy as np

# --- 固定随机种子，确保可复现 ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

# --- 路径配置 ---
RAW_TRAIN = "/mnt/data/prepared_data/sensevoice/train.jsonl"
VAL_JSONL = "/mnt/data/prepared_data/sensevoice/val.jsonl"
PREPARED_DIR = "/mnt/data/prepared_data/sv_ablation"
os.makedirs(PREPARED_DIR, exist_ok=True)

# --- 读取原始数据 ---
def load_jsonl(path):
    data = []
    with open(path) as f:
        for line in f:
            data.append(json.loads(line))
    return data

raw_train = load_jsonl(RAW_TRAIN)
print(f"原始训练集: {len(raw_train)} 条")

# --- 子采样函数 ---
def subsample(data, ratio, output_path):
    """随机采样 ratio 比例的数据，保存为 jsonl"""
    if ratio <= 0 or ratio > 1:
        raise ValueError(f"ratio must be in (0, 1], got {ratio}")
    n = max(1, int(len(data) * ratio))
    sampled = random.sample(data, n)
    with open(output_path, 'w') as f:
        for item in sampled:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"  子采样 {ratio:.0%}: {len(sampled)} 条 → {output_path}")
    return sampled

# --- 合成噪音增强 (A) ---
def add_gaussian_noise(audio_path, snr_db_range=(10, 20)):
    """加载音频，叠加高斯噪音，按随机 SNR"""
    wav, sr = torchaudio.load(audio_path)
    snr_db = random.uniform(*snr_db_range)
    signal_power = (wav ** 2).mean()
    noise_power = signal_power / (10 ** (snr_db / 10))
    noise = torch.randn_like(wav) * torch.sqrt(noise_power)
    noisy_wav = wav + noise
    return noisy_wav, sr

def generate_noise_a_dataset(data, output_dir, output_filename="train_noise_a.jsonl"):
    """为每条数据生成一份带噪音版本，保存到 output_dir/noise/ 目录"""
    noise_dir = os.path.join(output_dir, "noise_a")
    os.makedirs(noise_dir, exist_ok=True)
    jsonl_path = os.path.join(output_dir, output_filename)

    new_data = []
    for item in data:
        src = item['source']
        try:
            noisy_wav, sr = add_gaussian_noise(src)
            base = os.path.splitext(os.path.basename(src))[0]
            noisy_path = os.path.join(noise_dir, f"{base}_noise_a.wav")
            torchaudio.save(noisy_path, noisy_wav, sr)
            new_item = item.copy()
            new_item['source'] = noisy_path
            new_data.append(new_item)
        except Exception as e:
            print(f"  噪音处理失败: {src}, {e}")

    with open(jsonl_path, 'w') as f:
        for item in new_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"  合成噪音增强: {len(new_data)} 条 → {jsonl_path}")
    return jsonl_path

# --- 真实噪音增强 (B) ---
DEMAND_DIR = "/mnt/data/demand"
DEMAND_NOISES = []
if os.path.exists(DEMAND_DIR):
    DEMAND_NOISES = sorted([
        os.path.join(DEMAND_DIR, f)
        for f in os.listdir(DEMAND_DIR)
        if f.endswith('.wav')
    ])
print(f"  真实噪音文件: {len(DEMAND_NOISES)} 条")

def add_real_noise(audio_path, noise_path, snr_db_range=(10, 20)):
    """叠加真实噪音"""
    wav, sr = torchaudio.load(audio_path)
    noise, noise_sr = torchaudio.load(noise_path)
    if noise_sr != sr:
        noise = torchaudio.functional.resample(noise, noise_sr, sr)
    if noise.shape[1] < wav.shape[1]:
        pad = torch.zeros(wav.shape[1] - noise.shape[1])
        noise = torch.cat([noise, pad.unsqueeze(0)], dim=1)
    else:
        noise = noise[:, :wav.shape[1]]
    snr_db = random.uniform(*snr_db_range)
    signal_power = (wav ** 2).mean()
    noise_power = (noise ** 2).mean()
    if noise_power < 1e-10:
        noise_power = 1e-10
    adjusted_noise = noise * torch.sqrt(signal_power / (noise_power * (10 ** (snr_db / 10))))
    noisy_wav = wav + adjusted_noise
    return noisy_wav, sr

def generate_noise_b_dataset(data, output_dir, output_filename="train_noise_b.jsonl"):
    """使用 DEMAND 真实噪音增强"""
    if not DEMAND_NOISES:
        print("  警告: 未找到 DEMAND 噪音文件，跳过噪声B")
        return None
    noise_dir = os.path.join(output_dir, "noise_b")
    os.makedirs(noise_dir, exist_ok=True)
    jsonl_path = os.path.join(output_dir, output_filename)

    new_data = []
    for item in data:
        src = item['source']
        noise_path = random.choice(DEMAND_NOISES)
        try:
            noisy_wav, sr = add_real_noise(src, noise_path)
            base = os.path.splitext(os.path.basename(src))[0]
            noisy_path = os.path.join(noise_dir, f"{base}_noise_b.wav")
            torchaudio.save(noisy_path, noisy_wav, sr)
            new_item = item.copy()
            new_item['source'] = noisy_path
            new_data.append(new_item)
        except Exception as e:
            print(f"  噪音处理失败: {src}, {e}")

    with open(jsonl_path, 'w') as f:
        for item in new_data:
            f.write(json.dumps(item, ensure_ascii=False) + '\n')
    print(f"  真实噪音增强: {len(new_data)} 条 → {jsonl_path}")
    return jsonl_path

In [ ]:
print("\n数据生成完成!")
print(f"  25%:  {pct_configs[0][1]}")
print(f"  50%:  {pct_configs[1][1]}")
print(f"  75%:  {pct_configs[2][1]}")
print(f"  噪声A: {train_noise_a_path}")
print(f"  噪声B: {train_noise_b_path if train_noise_b_path else '(跳过: DEMAND噪音未找到)'}")

In [ ]:
import subprocess
import sys

def train_sensevoice_lora(train_jsonl, output_dir, exp_name):
    """训练 SenseVoice-Small LoRA"""
    os.makedirs(output_dir, exist_ok=True)

    cmd = [
        "torchrun", "--nproc_per_node=1", "-m", "funasr.bin.train_ds",
        "++model=iic/SenseVoiceSmall",
        f"++train_data_set_list={train_jsonl}",
        f"++valid_data_set_list={VAL_JSONL}",
        # data_split_num=1: 使用完整的 val.jsonl 作为验证集，不做额外分割
        "++dataset_conf.data_split_num=1",
        "++dataset_conf.batch_sampler=BatchSampler",
        "++dataset_conf.batch_size=6000",
        "++dataset_conf.sort_size=1024",
        "++dataset_conf.batch_type=token",
        "++dataset_conf.num_workers=4",
        "++train_conf.max_epoch=10",
        "++train_conf.log_interval=50",
        "++train_conf.validate_interval=2000",
        "++train_conf.save_checkpoint_interval=2000",
        "++train_conf.keep_nbest_models=5",
        "++train_conf.avg_nbest_model=3",
        "++train_conf.use_bf16=true",
        "++train_conf.grad_clip=5",
        "++lora_only=true",
        "++lora_bias=none",
        "++lora_rank=8",
        "++lora_alpha=16",
        "++lora_dropout=0.1",
        "++optim=adamw",
        "++optim_conf.lr=2e-4",
        "++optim_conf.weight_decay=0.0",
        f"++output_dir={output_dir}",
    ]

    print(f"\n{'='*60}")
    print(f"训练: {exp_name}")
    print(f"数据: {train_jsonl}")
    print(f"输出: {output_dir}")
    print(f"{'='*60}")

    process = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, bufsize=1,
    )
    for line in process.stdout:
        print(line, end='')
        sys.stdout.flush()

    returncode = process.wait()
    if returncode == 0:
        print(f"✓ {exp_name} 训练完成!")
    else:
        print(f"✗ {exp_name} 训练失败! 返回码: {returncode}")
    return returncode == 0

# --- 处理噪声数据路径（可能被前面cell跳过） ---
try:
    train_noise_a_path
except NameError:
    train_noise_a_path = None
try:
    train_noise_b_path
except NameError:
    train_noise_b_path = None

# --- 5 个实验组配置 ---
experiments = [
    ("25%",   "/mnt/data/prepared_data/sv_ablation/train_25pct.jsonl",   "/mnt/output/sv_25pct",   "sv_25pct"),
    ("50%",   "/mnt/data/prepared_data/sv_ablation/train_50pct.jsonl",   "/mnt/output/sv_50pct",   "sv_50pct"),
    ("75%",   "/mnt/data/prepared_data/sv_ablation/train_75pct.jsonl",   "/mnt/output/sv_75pct",   "sv_75pct"),
]
if train_noise_a_path:
    experiments.append(("噪声A", train_noise_a_path, "/mnt/output/sv_noise_a", "sv_noise_a"))
if train_noise_b_path:
    experiments.append(("噪声B", train_noise_b_path, "/mnt/output/sv_noise_b", "sv_noise_b"))

# --- 逐一训练（可跳过已完成的） ---
training_results = {}
for name, train_jsonl, output_dir, ckpt_name in experiments:
    ckpt_path = f"{output_dir}/model.pt.best"
    if os.path.exists(ckpt_path):
        print(f"\n[跳过] {name} 已完成，checkpoint 存在")
        training_results[name] = ckpt_path
        continue
    success = train_sensevoice_lora(train_jsonl, output_dir, f"{name} ({ckpt_name})")
    training_results[name] = ckpt_path if success else None

print("\n训练汇总:")
for name, ckpt in training_results.items():
    status = "✓" if ckpt and os.path.exists(ckpt) else "✗"
    print(f"  {status} {name}: {ckpt}")

In [ ]:
from funasr import AutoModel
import re
import time
import gc

def levenshtein(s1, s2):
    if len(s1) < len(s2):
        return levenshtein(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev = list(range(len(s2) + 1))
    for i, c1 in enumerate(s1):
        curr = [i + 1]
        for j, c2 in enumerate(s2):
            curr.append(min(prev[j + 1] + 1, curr[j] + 1, prev[j] + (c1 != c2)))
        prev = curr
    return prev[-1]

def clean_sensevoice_text(text):
    if not text:
        return ""
    return re.sub(r'<\|[^|]*\|>', '', text).strip()

def free_gpu():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

# 加载验证集
samples = []
with open(VAL_JSONL) as f:
    for line in f:
        samples.append(json.loads(line))
valid_samples = [s for s in samples if os.path.exists(s['source'])]
print(f"验证集: {len(samples)} 条, 有效: {len(valid_samples)} 条")

# 加载基座模型（用于 baseline 对比）
print("加载 SenseVoice 基座模型...")
sv_base = AutoModel(model="iic/SenseVoiceSmall", disable_update=True)

def eval_model(model, samples, label):
    """评估单个模型，返回 CER"""
    results = []
    total_cer, total_chars, exact = 0.0, 0, 0
    start = time.time()
    with torch.inference_mode():
        for i, s in enumerate(samples):
            audio, expected = s['source'], s['target']
            if not os.path.exists(audio):
                continue
            try:
                res = model.generate(input=audio, language="auto", use_itn=True)
                pred = clean_sensevoice_text(res[0]['text']) if res else ""
            except:
                pred = ""
            dist = levenshtein(expected, pred)
            ref_len = max(len(expected), 1)
            cer = dist / ref_len
            total_cer += dist
            total_chars += ref_len
            if expected == pred:
                exact += 1
            results.append({"id": i, "expected": expected, "predicted": pred, "cer": cer})
    cer = total_cer / total_chars if total_chars > 0 else 0
    elapsed = time.time() - start
    print(f"  {label}: CER={cer:.2%}, 精确={exact}/{len(results)}, 耗时={elapsed:.1f}s")
    return {"name": label, "cer": cer, "exact": exact, "total": len(results), "time": elapsed, "results": results}

# 基座模型评估
print("\n评估基座模型...")
sv_base_result = eval_model(sv_base, valid_samples, "SV-base")